# Najafi et al. (2020) on DANDI:000016, notebook 07c: learning, mice 4 and 3

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

Claims tested here, for mice 4 and 3 (the full table is in notebook 1; the three learning mice, 1, 2 and 4, are in notebooks 7a to 7c, and the attempt record combines them):

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `fig8ab-learning-accuracy` | Accuracy rises with learning and tracks performance | 07a, 07b, 07c | tested |
| `fig8ce-choice-signal-onset` | Choice signal onset moves earlier with learning | 07a, 07b, 07c | tested |
| `fig8de-learning-parallel` | E and I matched through learning | 07a, 07b, 07c | tested |
| `figs13c-equal-trials` | Learning increase with equal trial numbers | 07a, 07b, 07c | tested |
| `figs12-licking-running` | Licking and running do not explain learning | 07a, 07b, 07c | blocked in part (lick spout not identified); running part tested |
| `figs13ab-pseudo-learning` | Pseudo-population accuracy rises with learning | 07a, 07b, 07c | tested |

Mouse 3 (7 sessions) was not imaged through learning; it is run here because one reading of the
Figure 8B and 8C legends names it (claims file, fig8ab-learning-accuracy).

Decoders are trained per session on (a) all trials with a choice and (b) equal numbers of correct
and incorrect trials within each choice (claims file, fig8ab), at 0-97 ms before the choice and at
every bin. Performance is the fraction correct on easy trials. Early and late days are those below
the 20th or above the 80th percentile of performance: (a) every such day, (b) only the runs at the
start and end of training. For run time, 10 excitatory subsamples are used at the bin before the
choice, 1 at every bin (onsets) and 5 for pseudo populations, where the claims file has 50;
deviations, recorded.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
MICE_HERE = [4, 3]
A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files")
items = [(i, A.path[i], A.url[i]) for i in A.index if A.mouse[i] in MICE_HERE]
R = nj.map_sessions(S.decode_learning, items, workers=4)
binw = np.mean([np.diff(r["bin_times"]).mean() for r in R])
print(len(R), "sessions loaded")

## Data check

In [ ]:
rows = []
for r in R:
    row = dict(mouse=r["mouse"], date=r["date"], performance=r["performance"], inh=r["n_inh"], correct=r["n_correct"],
               bins=len(r["bin_times"]))
    for rd in ("a", "b"):
        x = r["readings"][rd]
        row[f"trials_{rd}"] = x["n_trials"]; row[f"ok_{rd}"] = x["ok"]
    rows.append(row)
D = pd.DataFrame(rows).sort_values(["mouse", "date"]).reset_index(drop=True)
print(D.groupby("mouse").agg(sessions=("date", "size"), first=("date", "min"), last=("date", "max"),
                             perf_min=("performance", "min"), perf_max=("performance", "max"),
                             trials_a_median=("trials_a", "median"), used_a=("ok_a", "sum"),
                             trials_b_median=("trials_b", "median"), used_b=("ok_b", "sum")).round(3).to_string())
print("Sessions not used (fewer than 10 trials of a choice):", D[~(D.ok_a & D.ok_b)][["mouse", "date", "trials_a", "trials_b"]].values.tolist())
w = pd.DataFrame([dict(step=r["wheel"]["t_step"], disp_max=np.nanmax(r["wheel"]["disp"]), mean_min=np.nanmin(r["wheel"]["mean"]),
                       mean_max=np.nanmax(r["wheel"]["mean"])) for r in R])
print("Wheel: step %.3f s; displacement over the bin 0 to %.3f; bin mean %.3f to %.3f (units not given)" %
      (w.step.median(), w.disp_max.max(), w.mean_min.min(), w.mean_max.max()))
EL = {}
for m in MICE_HERE:
    d = D[D.mouse == m]
    EL[m] = nj.early_late(d.date.values, d.performance.values)
    print(f"mouse {m}: 20th/80th percentiles of performance {EL[m]['thresholds'][0]:.3f}/{EL[m]['thresholds'][1]:.3f}; "
          f"early/late days (a) {EL[m]['a'][0].sum()}/{EL[m]['a'][1].sum()}, (b) {EL[m]['b'][0].sum()}/{EL[m]['b'][1].sum()}")
byd = {(r["mouse"], r["date"]): r for r in R}
def rows_of(m):
    return [byd[(m, d)] for d in D[D.mouse == m].date]

## fig8ab-learning-accuracy

> During learning, choice decoding accuracy increases and correlates with behavioural performance
> across sessions, for all, excitatory and inhibitory populations (r = 0.55, 0.35, 0.32 in mouse 2
> and 0.57, 0.63, 0.32 in mouse 3).

Criterion: a decoder per session at 0-97 ms before the choice; trials (a) all with a choice and (b)
equal correct and incorrect within each choice; Pearson r of accuracy against performance across
sessions. The legend's mice 2 and 3 are read as the second and third learning mice (mice 2 and 4)
or as the paper's mice 2 and 3; the attempt record compares the values. Here: r and p per mouse,
population and trial reading.

In [ ]:
rows = []
for m in MICE_HERE:
    rr = rows_of(m)
    for rd in ("a", "b"):
        use = [r for r in rr if r["readings"][rd]["ok"]]
        perf = np.array([r["performance"] for r in use])
        for pop in ("all", "exc", "inh"):
            acc = np.array([r["readings"][rd]["bin1"][pop] for r in use])
            sh = np.array([r["readings"][rd]["bin1"][pop + "_shfl"] for r in use])
            rr_, p_ = stats.pearsonr(acc, perf) if len(use) > 2 else (np.nan, np.nan)
            rows.append(dict(mouse=m, trials=rd, population=pop, sessions=len(use), r=rr_, p=p_, acc_mean=acc.mean(),
                             acc_min=acc.min(), acc_max=acc.max(), shfl_mean=sh.mean()))
T8ab = pd.DataFrame(rows)
print(T8ab.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("r > 0 with p < 0.05 for all three populations: ", {(m, rd): bool(((g.r > 0) & (g.p < 0.05)).all())
                                                        for (m, rd), g in T8ab.groupby(["mouse", "trials"])})

**Result: (to be written)**

## fig8ce-choice-signal-onset

> The choice signal emerges progressively earlier in the trial as mice become experts, so the
> onset of above-chance decoding correlates negatively with performance (r = -0.60, -0.34, -0.38 in
> mouse 2 and -0.60, -0.27, -0.28 in mouse 3 for all, excitatory and inhibitory).

Criterion: decoders at every bin; per session and bin, accuracy against the shuffle by a t-test
across the 50 repetitions; the onset is the start of the first run of significant bins lasting at
least 500 ms (5 bins), in ms from the choice; sessions without such a run left out. Alpha 0.05 of
the Methods, 0.001 of the code as a variant. Significance counts only bins where accuracy is above
the shuffle. Pearson r of onset against performance.

In [ ]:
def onset(pb, times, alpha):
    sig = (pb > 0) & (pb < alpha)
    run = 0
    for k, s_ in enumerate(sig):
        run = run + 1 if s_ else 0
        if run == 5:
            return times[k - 4]
    return np.nan
rows = []
ONS = {}
for m in MICE_HERE:
    rr = rows_of(m)
    for rd in ("a", "b"):
        use = [r for r in rr if r["readings"][rd]["ok"]]
        for alpha in (0.05, 0.001):
            for pop in ("all", "exc", "inh"):
                on = np.array([onset(r["readings"][rd]["p_bins"][pop], r["bin_times"], alpha) for r in use])
                perf = np.array([r["performance"] for r in use])
                ONS[(m, rd, alpha, pop)] = (on, perf, [r["date"] for r in use])
                ok = np.isfinite(on)
                rr_, p_ = stats.pearsonr(on[ok], perf[ok]) if ok.sum() > 2 else (np.nan, np.nan)
                rows.append(dict(mouse=m, trials=rd, alpha=alpha, population=pop, sessions=len(use), with_onset=int(ok.sum()),
                                 r=rr_, p=p_, onset_median_ms=np.nanmedian(on) if ok.any() else np.nan,
                                 onset_min=np.nanmin(on) if ok.any() else np.nan, onset_max=np.nanmax(on) if ok.any() else np.nan))
T8ce = pd.DataFrame(rows)
print(T8ce.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("r < 0 with p < 0.05 for all three populations:", {(m, rd, a): bool(((g.r < 0) & (g.p < 0.05)).all())
                                                       for (m, rd, a), g in T8ce.groupby(["mouse", "trials", "alpha"])})

**Result: (to be written)**

## fig8de-learning-parallel

> The choice signal emerges at the same time in excitatory and inhibitory populations, and its
> magnitude and timing are matched for the two cell types throughout learning.

Criterion: early and late days (readings a and b); accuracy (0-97 ms before the choice) and onset of
inhibitory and subsampled excitatory decoders in early and in late days; unpaired two-tailed t-test
across days at p < 0.05, a claim of absence. Reproduced when, in every learning mouse, neither
accuracy nor onset differs between the types in early or late days, and late accuracy exceeds early
accuracy for both types by the same test. Shown for both trial readings and, for onsets, alpha 0.05.

In [ ]:
rows = []
for m in MICE_HERE:
    rr = rows_of(m)
    dates = D[D.mouse == m].date.values
    for days in ("a", "b"):
        e_mask, l_mask = EL[m][days]
        early, late = set(dates[e_mask]), set(dates[l_mask])
        for rd in ("a", "b"):
            use = [r for r in rr if r["readings"][rd]["ok"]]
            def vals(pop, which):
                sel = early if which == "early" else late
                return np.array([r["readings"][rd]["bin1"][pop] for r in use if r["date"] in sel])
            on = {pop: dict(zip(ONS[(m, rd, 0.05, pop)][2], ONS[(m, rd, 0.05, pop)][0])) for pop in ("exc", "inh")}
            row = dict(mouse=m, days=days, trials=rd, n_early=len(vals("inh", "early")), n_late=len(vals("inh", "late")))
            for which in ("early", "late"):
                ve, vi = vals("exc", which), vals("inh", which)
                row[f"acc_exc_{which}"], row[f"acc_inh_{which}"] = (ve.mean() if len(ve) else np.nan), (vi.mean() if len(vi) else np.nan)
                row[f"p_acc_{which}"] = nj.ttest(ve, vi)[1]
                sel = early if which == "early" else late
                oe = np.array([on["exc"][d] for d in sel if d in on["exc"]]); oi = np.array([on["inh"][d] for d in sel if d in on["inh"]])
                row[f"onset_exc_{which}"], row[f"onset_inh_{which}"] = np.nanmean(oe) if np.isfinite(oe).any() else np.nan, np.nanmean(oi) if np.isfinite(oi).any() else np.nan
                row[f"p_onset_{which}"] = nj.ttest(oe, oi)[1]
            row["p_late_gt_early_exc"] = nj.ttest(vals("exc", "late"), vals("exc", "early"))[1]
            row["p_late_gt_early_inh"] = nj.ttest(vals("inh", "late"), vals("inh", "early"))[1]
            rows.append(row)
T8de = pd.DataFrame(rows)
print(T8de.T.to_string(float_format=lambda v: f"{v:.4g}"))
for _, r_ in T8de.iterrows():
    no_diff = all(not (r_[k] < 0.05) for k in ("p_acc_early", "p_acc_late", "p_onset_early", "p_onset_late"))
    rise = (r_.acc_exc_late > r_.acc_exc_early and r_.p_late_gt_early_exc < 0.05 and
            r_.acc_inh_late > r_.acc_inh_early and r_.p_late_gt_early_inh < 0.05)
    print(f"mouse {r_.mouse}, days {r_.days}, trials {r_.trials}: no E-I difference {no_diff}; late above early for both {rise}")

**Result: (to be written)**

## figs13c-equal-trials

> The learning-related increase in decoding accuracy remains when the number of trials is equal in
> every session.

Criterion: decoders trained on correct trials only, with the same number of trials in every session
of a mouse (the minimum over its sessions); late accuracy above early accuracy for all neurons by
the unpaired two-tailed t-test across days at p < 0.05; early and late days as fig8de.

In [ ]:
rows = []
for m in MICE_HERE:
    rr = rows_of(m)
    n = min(min((r["y1_correct"] == 1).sum(), (r["y1_correct"] == 0).sum()) for r in rr)
    res = nj.map_sessions(S.decode_equal_trials, [(r["idx"], r["X1_correct"], r["y1_correct"], int(n)) for r in rr], workers=4)
    acc = {r["date"]: x["acc"] for r, x in zip(rr, res)}
    dates = D[D.mouse == m].date.values
    for days in ("a", "b"):
        e = np.array([acc[d] for d in dates[EL[m][days][0]]]); l = np.array([acc[d] for d in dates[EL[m][days][1]]])
        tt = nj.ttest(l, e)
        rows.append(dict(mouse=m, trials_per_choice=int(n), days=days, early=e.mean() if len(e) else np.nan,
                         late=l.mean() if len(l) else np.nan, n_early=len(e), n_late=len(l), p=tt[1]))
    print(f"mouse {m}: accuracy per session with {2 * n} correct trials, %.1f to %.1f%%" % (min(acc.values()), max(acc.values())))
T13c = pd.DataFrame(rows)
print(T13c.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: (to be written)**

## figs13ab-pseudo-learning

> Decoding accuracy of pseudo populations, which lack noise correlations, also increases
> significantly with training, so reduced noise correlations do not alone explain the improved
> accuracy.

Criterion: pseudo populations (trial order shuffled per neuron within each choice) for all,
subsampled excitatory and inhibitory decoders at 0-97 ms before the choice, trials as fig8ab; early
and late days as fig8de. Reproduced when late exceeds early by the unpaired two-tailed t-test at
p < 0.05 in every learning mouse for the all-neuron decoder, and in the same direction for both
types.

In [ ]:
rows = []
for m in MICE_HERE:
    rr = rows_of(m)
    dates = D[D.mouse == m].date.values
    for days in ("a", "b"):
        early, late = set(dates[EL[m][days][0]]), set(dates[EL[m][days][1]])
        for rd in ("a", "b"):
            use = [r for r in rr if r["readings"][rd]["ok"]]
            row = dict(mouse=m, days=days, trials=rd)
            for pop in ("all", "exc", "inh"):
                e = np.array([r["readings"][rd]["pseudo"][pop] for r in use if r["date"] in early])
                l = np.array([r["readings"][rd]["pseudo"][pop] for r in use if r["date"] in late])
                row[f"{pop}_early"], row[f"{pop}_late"] = (e.mean() if len(e) else np.nan), (l.mean() if len(l) else np.nan)
                row[f"p_{pop}"] = nj.ttest(l, e)[1]
            row["n_early"], row["n_late"] = len(e), len(l)
            rows.append(row)
T13ab = pd.DataFrame(rows)
print(T13ab.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: (to be written)**

## figs12-licking-running

> Licking and running do not explain the learning-related changes; centre-spout licks before left
> and right choices are similar early and late in training, and decoding remains accurate in
> sessions where running did not differ between choices.

The lick part needs licks to the centre spout; the lick trace of the files is one binary series
without the spout (data check of notebook 1), so it is shown here only as a variant on licks to any
spout (250 ms before the choice, 10 ms bins, 1 ms as a variant): Pearson r of the left-choice and
right-choice mean lick-rate traces per session, early against late days. Running: the wheel trace in
the bin 0-97 ms before the choice, read (i) as the distance turned over the bin (absolute change of
the trace) and (ii) as the mean of the trace over the bin, since the files do not say what the trace
holds; left against right by a t-test per session; on sessions with P > 0.05, all-neuron accuracy
against the shuffle by the unpaired t-test across those sessions.

In [ ]:
rows = []
for m in MICE_HERE:
    rr = rows_of(m)
    dates = D[D.mouse == m].date.values
    for measure in ("disp", "mean"):
        sel_dates = []
        for r in rr:
            w_, y_, ok_ = r["wheel"][measure], r["wheel"]["choice"], r["wheel"]["ok"] & np.isfinite(r["wheel"][measure])
            p_ = stats.ttest_ind(w_[ok_ & (y_ == 1)], w_[ok_ & (y_ == 0)]).pvalue
            if p_ > 0.05:
                sel_dates.append(r["date"])
        for rd in ("a", "b"):
            use = [r for r in rr if r["readings"][rd]["ok"] and r["date"] in sel_dates]
            acc = np.array([r["readings"][rd]["bin1"]["all"] for r in use]); sh = np.array([r["readings"][rd]["bin1"]["all_shfl"] for r in use])
            rows.append(dict(mouse=m, running=measure, trials=rd, sessions_p_above_05=len(sel_dates), used=len(use),
                             acc=acc.mean() if len(use) else np.nan, shfl=sh.mean() if len(use) else np.nan, p=nj.ttest(acc, sh)[1]))
T12 = pd.DataFrame(rows)
print(T12.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
# lick variant (any spout)
rows = []
for m in MICE_HERE:
    rr = rows_of(m)
    dates = D[D.mouse == m].date.values
    for width in (10, 1):
        rs = {}
        for r in rr:
            t_, x_, y_, ok_ = r["lick"]["t"], r["lick"]["x"], r["lick"]["choice"], r["lick"]["ok"]
            nb = int(round(250 / width))
            k = np.clip(((t_ + 0.25) * 1000 // width).astype(int), 0, nb - 1)
            tr_ = np.stack([np.bincount(k, weights=np.nan_to_num(x_[:, j]), minlength=nb) for j in range(x_.shape[1])], 1)
            a1, a0 = tr_[:, ok_ & (y_ == 1)].mean(1), tr_[:, ok_ & (y_ == 0)].mean(1)
            rs[r["date"]] = stats.pearsonr(a1, a0)[0] if a1.std() > 0 and a0.std() > 0 else np.nan
        for days in ("a", "b"):
            e = np.array([rs[d] for d in dates[EL[m][days][0]]]); l = np.array([rs[d] for d in dates[EL[m][days][1]]])
            rows.append(dict(mouse=m, bin_ms=width, days=days, r_early=np.nanmean(e) if len(e) else np.nan,
                             r_late=np.nanmean(l) if len(l) else np.nan, p=nj.ttest(e, l)[1]))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: (to be written)**

## Summary

(to be written)